In [1]:
import numpy as np
import pandas as pd
import os
import joblib 
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [3]:
#housing[housing["Area"] == housing["Area"].max()] for knowing the maximum value

MODEL_FILE = "model.pkl"
PIPELINE = "pipeline.pkl"

#build the pipeline
def build_pipeline():
    num_pipeline = Pipeline([
        ("imputer",SimpleImputer(strategy = "median")),
        ("scaler",StandardScaler())
    ])
    return num_pipeline

if not os.path.exists(MODEL_FILE):
    housing = pd.read_excel("final_dataset.xlsx")
    df = housing.iloc[:,1:-37]
    housing["area_cat"] = pd.cut(
        housing["Area"],
        bins = [0,1000,2000,3000,5000,np.inf],
        labels = [1,2,3,4,5]
    )
    #split the dataset into training and test
    split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
    for train_index,test_index in split.split(housing,housing["area_cat"]):
        strat_train_set = housing.loc[train_index].drop("area_cat",axis=1)
        strat_test_set = housing.loc[test_index].drop("area_cat",axis=1)

    #work with the copy of the training dataset
    train_set_copy = strat_train_set.copy()
    #test_set_copy =strat_test_set.copy()  # just to see the count
    #test = test_set_copy.copy()
    #test.count

    #creating an Input csv file 
    test_set = strat_test_set.copy()
    test_set.to_csv("input.csv",index=False)

    # separate the predictors and labels
    labels = train_set_copy["Price"].copy()
    features = train_set_copy.drop("Price",axis=1)

    #if there is any categorical and numerical data then separate it before giving it to model
    num_pipeline = Pipeline([
        ("imputer",SimpleImputer(strategy = "median")),
        ("scaler",StandardScaler())
    ])
    #transform the data
    housing_prepared = num_pipeline.fit_transform(train_set_copy)

    #Train the model
    model = RandomForestRegressor(random_state=42)
    model.fit(housing_prepared,labels)

    #save model & pipeline
    joblib.dump(model,MODEL_FILE)
    joblib.dump(num_pipeline,PIPELINE)
    print("Model trained and saved.")
else: #inference phase
    model = joblib.load(MODEL_FILE)
    pipeline = num_pipeline
    input_data = pd.read_csv("input.csv")
    transformed_input = pipeline.transform(input_data)
    predictions = model.predict(transformed_input)
    input_data["median_house_value"] = predictions
    input_data.to_csv("output.csv",index = False)
    print("Inference Complete. Results Saved to Output.csv")
    
    
    


Inference Complete. Results Saved to Output.csv
